<div style="background:linear-gradient(135deg,#1A5276 0%,#2E86C1 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#AED6F1;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 2 · CLASE 7</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Ejercicios: Scoring, Threshold y Comparación de Modelos</h1>
  <p style="color:#AED6F1;font-size:16px;margin:0 0 24px 0;font-style:italic;">Aplicar logística en contexto de negocio real</p>
  <hr style="border-color:#5DADE2;margin:0 0 20px 0;">
  <p style="color:#EBF5FB;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; <strong>Duración:</strong> 2 horas</p>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import expit
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
SEED=42; np.random.seed(SEED)
print('✅ Setup listo')

---
## 🟢 Ejercicio 1 — Threshold por costo de negocio

Dado un modelo de detección de fraude:
- FN = fraude no detectado = -5,000 USD
- FP = alerta falsa = -50 USD

1. Calcular el costo total para thresholds de 0.1 a 0.9
2. Encontrar el threshold óptimo
3. Graficar costo vs threshold

In [ ]:
np.random.seed(SEED); n=1000
X_fr=StandardScaler().fit_transform(np.random.randn(n,3))
y_fr=np.random.binomial(1,expit(np.column_stack([np.ones(n),X_fr])@np.array([-2,1.8,-1.2,0.7])))
Xtr_f,Xte_f,ytr_f,yte_f=train_test_split(X_fr,y_fr,test_size=0.25,random_state=SEED)
lr_f=LogisticRegression(C=1e6,solver='lbfgs',max_iter=500); lr_f.fit(Xtr_f,ytr_f)
proba_f=lr_f.predict_proba(Xte_f)[:,1]
# --- Tu código aquí ---

In [ ]:
# ✅ SOLUCIÓN
from sklearn.metrics import confusion_matrix
np.random.seed(SEED); n=1000
X_fr=StandardScaler().fit_transform(np.random.randn(n,3))
y_fr=np.random.binomial(1,expit(np.column_stack([np.ones(n),X_fr])@np.array([-2,1.8,-1.2,0.7])))
Xtr_f,Xte_f,ytr_f,yte_f=train_test_split(X_fr,y_fr,test_size=0.25,random_state=SEED)
lr_f=LogisticRegression(C=1e6,solver='lbfgs',max_iter=500); lr_f.fit(Xtr_f,ytr_f)
proba_f=lr_f.predict_proba(Xte_f)[:,1]
COSTO_FN=5000; COSTO_FP=50
ths=np.arange(0.1,0.9,0.05); costos_f=[]
for th in ths:
    yp=(proba_f>=th).astype(int)
    tn,fp,fn,tp=confusion_matrix(yte_f,yp).ravel()
    costos_f.append(fn*COSTO_FN+fp*COSTO_FP)
opt=ths[np.argmin(costos_f)]
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(ths,costos_f,'o-',color='#E74C3C',lw=2); ax.axvline(opt,color='#27AE60',lw=2,linestyle='--',label=f'Óptimo={opt:.2f}')
ax.set(xlabel='Threshold',ylabel='Costo (USD)',title='Threshold óptimo por costo de negocio')
ax.legend(); ax.grid(True,alpha=0.2); plt.tight_layout(); plt.show()
print(f'Threshold óptimo: {opt:.2f}  Costo mínimo: ${min(costos_f):,.0f}')

---
## 🟡 Ejercicio 2 — Logística vs LDA: cuándo difieren

Generar datos con distribución no normal (ej. exponencial) y comparar logística vs LDA.
¿Cuál tiene mayor AUC? ¿Por qué?

In [ ]:
np.random.seed(SEED); n=600
x1=np.random.exponential(2,n); x2=np.random.gamma(3,1,n)
X_nd=StandardScaler().fit_transform(np.column_stack([x1,x2]))
y_nd=np.random.binomial(1,expit(1.5*X_nd[:,0]-X_nd[:,1]-0.5))
Xtr_nd,Xte_nd,ytr_nd,yte_nd=train_test_split(X_nd,y_nd,test_size=0.25,random_state=SEED)
# --- Comparar LogisticRegression vs LinearDiscriminantAnalysis ---

In [ ]:
# ✅ SOLUCIÓN
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
np.random.seed(SEED); n=600
x1=np.random.exponential(2,n); x2=np.random.gamma(3,1,n)
X_nd=StandardScaler().fit_transform(np.column_stack([x1,x2]))
y_nd=np.random.binomial(1,expit(1.5*X_nd[:,0]-X_nd[:,1]-0.5))
Xtr_nd,Xte_nd,ytr_nd,yte_nd=train_test_split(X_nd,y_nd,test_size=0.25,random_state=SEED)
lr_nd=LogisticRegression(C=1e6,solver='lbfgs',max_iter=500); lr_nd.fit(Xtr_nd,ytr_nd)
lda_nd=LinearDiscriminantAnalysis(); lda_nd.fit(Xtr_nd,ytr_nd)
for nm,m in [('Logística',lr_nd),('LDA',lda_nd)]:
    auc=roc_auc_score(yte_nd,m.predict_proba(Xte_nd)[:,1])
    acc=accuracy_score(yte_nd,m.predict(Xte_nd))
    print(f'  {nm:10s}: AUC={auc:.4f}  Acc={acc:.4f}')
print('\n→ Con distribución no normal, Logística tiende a tener mayor AUC.')

---
## 🔴 Ejercicio 3 — Sistema de scoring completo

Desarrollar un sistema completo con:
1. Entrenamiento y evaluación de 3 modelos (Logística, LDA, QDA)
2. Selección por AUC
3. Threshold óptimo del mejor modelo
4. Reporte final con predicciones

In [ ]:
np.random.seed(SEED); n=2000
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
edad=np.random.randint(20,70,n).astype(float); ingr=np.random.normal(40000,15000,n)
deuda=np.random.uniform(0.1,0.8,n); hist=np.random.choice([0,1,2],n,p=[0.6,0.3,0.1]).astype(float)
logit_s=-2.5+0.02*edad-0.00002*ingr+2.8*deuda+1.2*hist
y_sc=np.random.binomial(1,expit(logit_s))
X_sc2=StandardScaler().fit_transform(np.column_stack([edad,ingr,deuda,hist]))
Xtr_s,Xte_s,ytr_s,yte_s=train_test_split(X_sc2,y_sc,test_size=0.2,random_state=SEED)
# --- Tu pipeline de 3 modelos aquí ---

In [ ]:
# ✅ SOLUCIÓN
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
np.random.seed(SEED); n=2000
edad=np.random.randint(20,70,n).astype(float); ingr=np.random.normal(40000,15000,n)
deuda=np.random.uniform(0.1,0.8,n); hist=np.random.choice([0,1,2],n,p=[0.6,0.3,0.1]).astype(float)
logit_s=-2.5+0.02*edad-0.00002*ingr+2.8*deuda+1.2*hist; y_sc=np.random.binomial(1,expit(logit_s))
X_sc2=StandardScaler().fit_transform(np.column_stack([edad,ingr,deuda,hist]))
Xtr_s,Xte_s,ytr_s,yte_s=train_test_split(X_sc2,y_sc,test_size=0.2,random_state=SEED)
modelos_s={'Logística':LogisticRegression(C=1e6,solver='lbfgs',max_iter=500),
           'LDA':LinearDiscriminantAnalysis(),'QDA':QuadraticDiscriminantAnalysis()}
resultados_s={}
for nm,m in modelos_s.items():
    m.fit(Xtr_s,ytr_s); proba_s=m.predict_proba(Xte_s)[:,1]
    auc_s=roc_auc_score(yte_s,proba_s)
    resultados_s[nm]=(auc_s,proba_s); print(f'  {nm:12s}: AUC={auc_s:.4f}')
mejor_nm=max(resultados_s,key=lambda k:resultados_s[k][0])
mejor_proba=resultados_s[mejor_nm][1]
from sklearn.metrics import confusion_matrix
ths=np.arange(0.1,0.9,0.05)
J=[stats.norm.cdf(0)-0.5+roc_curve(yte_s,mejor_proba)[1][np.argmin(np.abs(roc_curve(yte_s,mejor_proba)[2]-th))] for th in ths]
fpr_r,tpr_r,thr_r=roc_curve(yte_s,mejor_proba)
J_r=tpr_r-fpr_r; opt_th_s=thr_r[np.argmax(J_r)]
print(f'\nMejor modelo: {mejor_nm}  Threshold óptimo: {opt_th_s:.3f}')
y_final=(mejor_proba>=opt_th_s).astype(int)
print(classification_report(yte_s,y_final,target_names=['No default','Default']))

<div style="background:#1A5276;color:white;padding:20px 24px;border-radius:8px;"><strong>Próxima clase</strong><br>Módulo 3 — Análisis Discriminante · LDA · Fisher · Frontera de decisión<br><em>Docente: Josef Rodriguez · Curso 8</em></div>